In [1]:
# --- Бібліотеки для даної роботи ---
try:
    import numpy, pandas, matplotlib, plotly, nbformat, sklearn, jinja2
    print("Бібліотеки вже встановлені. Пропускаємо інсталяцію.")
except ImportError:
    print("Встановлюємо бібліотеки...")
    %pip install numpy pandas matplotlib plotly nbformat scikit-learn jinja2

Бібліотеки вже встановлені. Пропускаємо інсталяцію.


## Домашнє завдання: Тема 6. Ряди та аналіз Фур'є в математиці

### **Це допоможе закріпити такі навички:**

- Обчислення перетворення Фур'є
- Застосування швидкого перетворення Фур'є при обробці звуку з метою генерації ознак, які далі можуть використовуватись для кластеризації чи класифікації

### **Завдання (крок за кроком):**

***Для цієї задачі необхідно буде завантажити набір даних ESC-50, як це було зроблено в конспекті.***

1. **Зменшити обсяг даних:**
   - Зробити вибірку звуків із мітками `'dog'` та `'chirping_birds'`.

2. **Згенерувати матрицю спектрограми:**
   - За допомогою наведеної в конспекті функції `spectrogram`.

3. **Узагальнити та зменшити розмір спектрограми:**
   - Використати функцію `pooling`.

4. **Перетворити матрицю спектрограми у вектор для подальшого спектрального аналізу:**
   - За допомогою методу `flaten()`.

5. **Кластеризація отриманих даних:**
   - Використати функцію `SpectralClustering` бібліотеки `sklearn` [(аналогічно до ДЗ теми №1)](https://github.com/IRONKAGE/goit-num-hw-01).

6. **Проаналізувати отримані кластери:**
   - Чи потрапили в різні кластери звуки різного походження?

7. **Висновок:**
   - Зробити висновок про значення застосування перетворення Фур'є для вилучення ознак даних.

**0. Імпорт необхідних бібліотек:**

In [8]:
import warnings
warnings.filterwarnings('ignore')

import os
import sys
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.cluster import SpectralClustering
from IPython.display import display

import urllib.request
import zipfile
import shutil
from scipy.io import wavfile

**0.3. Завантаження або генерація набору даних (ESC-50):**

In [ ]:
url = "https://github.com/karolpiczak/ESC-50/archive/master.zip"
zip_path = "ESC-50.zip"

def generate_mock_dataset():
    print("⚠️ Мережева помилка. Генеруємо синтетичний набір даних для тестування (Mock Data)...")
    
    os.makedirs("audio", exist_ok=True)
    
    sr = 22050
    duration = 2
    t = np.linspace(0, duration, sr * duration, False)
    
    filenames = []
    categories = []
    
    for i in range(3):
        fname = f"mock_dog_{i}.wav"
        wave = np.sin(2 * np.pi * 300 * t) * np.where(np.sin(2 * np.pi * 3 * t) > 0, 1, 0)
        wavfile.write(os.path.join("audio", fname), sr, wave.astype(np.float32))
        filenames.append(fname)
        categories.append('dog')
        
    for i in range(3):
        fname = f"mock_bird_{i}.wav"
        wave = np.sin(2 * np.pi * 4000 * t) * np.where(np.sin(2 * np.pi * 10 * t) > 0.5, 1, 0)
        wavfile.write(os.path.join("audio", fname), sr, wave.astype(np.float32))
        filenames.append(fname)
        categories.append('chirping_birds')

    df = pd.DataFrame({'filename': filenames, 'category': categories})
    df.to_csv("esc50.csv", index=False)
    print("✅ Синтетичний набір даних (Mock ESC-50) успішно створено!")

def download_with_resume(url, filename):
    existing_size = 0
    if os.path.exists(filename):
        existing_size = os.path.getsize(filename)
        
    req = urllib.request.Request(url)
    if existing_size > 0:
        req.add_header("Range", f"bytes={existing_size}-")
        
    try:
        with urllib.request.urlopen(req) as response:
            total_size = response.headers.get('Content-Length')
            total_size = int(total_size) + existing_size if total_size else 0
            
            domain = urllib.parse.urlparse(url).netloc
            
            if response.getcode() == 206:
                print(f"🔄 Знайдено незавершений файл ({existing_size / (1024*1024):.1f} MB). Продовжуємо завантаження...")
                mode = 'ab' # Append binary
            else:
                if existing_size > 0:
                    print(f"⚠️ Сервер ({domain}) не підтримує дозавантаження. Починаємо запис спочатку...")
                mode = 'wb' # Write binary
                existing_size = 0
            
            with open(filename, mode) as f:
                while True:
                    chunk = response.read(8192)
                    if not chunk:
                        break
                    f.write(chunk)
                    existing_size += len(chunk)
                    
                    if total_size > 0:
                        percent = min(100, existing_size * 100 / total_size)
                        bar_length = 40
                        filled_length = int(bar_length * percent // 100)
                        bar = '█' * filled_length + '-' * (bar_length - filled_length)
                        sys.stdout.write(f'\r⏳ Завантаження: [{bar}] {percent:.1f}% ({existing_size / (1024*1024):.1f} MB / {total_size / (1024*1024):.1f} MB)')
                    else:
                        sys.stdout.write(f'\r⏳ Завантаження: {existing_size / (1024*1024):.1f} MB (реальний розмір невідомий)...')
                    sys.stdout.flush()
        print()
    except Exception as e:
        raise e

def is_valid_zip(filepath):
    if not os.path.exists(filepath):
        return False
    if not zipfile.is_zipfile(filepath):
        return False
    try:
        with zipfile.ZipFile(filepath, 'r') as z:
            if z.testzip() is not None:
                return False
    except Exception:
        return False
    return True

if not os.path.exists("esc50.csv") or not os.path.exists("audio"):
    print("🛋️ Починаємо налаштування середовища...")
    try:
        if os.path.exists(zip_path):
            print(f"🔍 Знайдено локальний архів '{zip_path}'. Перевіряємо цілісність...")
            if is_valid_zip(zip_path):
                print("✅ Архів цілий. Пропускаємо завантаження.")
            else:
                print("⚠️ Архів недозавантажений або пошкоджений. Спробуємо дозавантажити або перезаписати...")
                download_with_resume(url, zip_path)
        else:
            download_with_resume(url, zip_path)

        if not is_valid_zip(zip_path):
            raise Exception("💔 Завантажений архів виявився пошкодженим після всіх спроб...")
            
        print("📦 Розпаковуємо архів...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall("temp_esc50")
            
        print("⚙️ Налаштовуємо структуру файлів...")
        if not os.path.exists("esc50.csv"):
            shutil.move("temp_esc50/ESC-50-master/meta/esc50.csv", "esc50.csv")
        if not os.path.exists("audio"):
            shutil.move("temp_esc50/ESC-50-master/audio", "audio")
            
        if os.path.exists("temp_esc50"):
            shutil.rmtree("temp_esc50")
            
        print("✅ Успіх! Файл 'esc50.csv' та папка 'audio/' готові до роботи.")
        
    except Exception as e:
        print(f"\n❌ Помилка: {e}")
        generate_mock_dataset()
else:
    print("🧸 Набір даних вже знайдено у вашій директорії. Можна продовжувати!")

🧸 Набір даних вже знайдено у вашій директорії. Можна продовжувати!


**0.5. Конфігурація експерименту (Глобальні змінні):**

**1. Зменшити обсяг даних:**

**2. Згенерувати матрицю спектрограми:**

**3. Узагальнити та зменшити розмір спектрограми:**

**4. Перетворити матрицю спектрограми у вектор для подальшого спектрального аналізу:**

**5. Кластеризація отриманих даних:**

**6. Проаналізувати отримані кластери:**

**7. Висновок:**